In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [2]:
X1 = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\version_4\X_train_road_4_1.csv')
S1 = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\version_4\X_train_speed_4_1.csv')
y1 = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\version_4\y_train_wheel_4_1.csv')

In [3]:
X2 = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\version_4\X_train_road_4_2.csv')
S2 = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\version_4\X_train_speed_4_2.csv')
y2 = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\version_4\y_train_wheel_4_2.csv')

In [4]:
X = pd.concat([X1, X2], axis=0)
S = pd.concat([S1, S2], axis=0)
y = pd.concat([y1, y2], axis=0)

In [5]:
X_train = torch.tensor(X.values, dtype=torch.float32).to(device='cuda')
S_train = torch.tensor(S.values / 170, dtype=torch.float32).to(device='cuda')
y_tensor = torch.tensor(y.values, dtype=torch.float32).to(device='cuda')

In [6]:
print(len(X_train))
print(len(S_train))
print(len(y_tensor))

30637
30637
30637


In [7]:
X_tensor = torch.cat((X_train, S_train), dim=1)
print(X_tensor[0])
print(X_tensor[-1])
print(X_tensor[0].size())

tensor([0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.4882], device='cuda:0')
tensor([0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.7059], device='cuda:0')
torch.Size([3073])


In [8]:
print(y_tensor[0])
print(y_tensor[0].size())

tensor([-0.0044, -0.0028], device='cuda:0')
torch.Size([2])


In [9]:
dataset = TensorDataset(X_tensor, y_tensor)
data_loader = DataLoader(dataset, batch_size=512, shuffle=False)

In [10]:
class FeedforwardNet(nn.Module):
    def __init__(self, input_size=3073, hidden_size_1=1024, hidden_size_2=512, hidden_size_3=512,
                 output_size=2):
        super(FeedforwardNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size_1)
        self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
        self.fc3 = nn.Linear(hidden_size_2, hidden_size_3)
        self.fc4 = nn.Linear(hidden_size_3, output_size)
        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()

    def forward(self, input_data):
        out = self.fc1(input_data)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        out = self.relu(out)
        out = self.fc4(out)
        out = self.tanh(out)
        return out

In [11]:
model = FeedforwardNet().cuda()

In [12]:
# Функция для расчета L1 штрафа
def l1_penalty(params):
    return sum(p.abs().sum() for p in params)

# Функция для расчета L2 штрафа
def l2_penalty(params):
    return sum(p.pow(2.0).sum() for p in params)

# Обучение модели с L1 и L2 регуляризацией
criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=0.000001, weight_decay=0.0001)
l1_factor = 1e-5
l2_factor = 1e-5


In [13]:
# Обучение модели
num_epochs = 100
for epoch in range(num_epochs):
    for batch_x, batch_y in data_loader:  # Итерация по батчам данных
        optimizer.zero_grad()  # Обнуление градиентов

        outputs = model(batch_x.unsqueeze(0))  # Передача входных данных через модель
        loss = criterion(outputs, batch_y)  # Вычисление потерь
        
        l1_loss = l1_penalty(model.parameters())
        l2_loss = l2_penalty(model.parameters())
        total_loss = loss + l1_factor * l1_loss + l2_factor * l2_loss
        
        loss.backward()  # Обратное распространение ошибки
        optimizer.step()  # Обновление весов модели

    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.9f}')

C:\Users\filip\anaconda3\envs\ETS_autopylot\Lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([512, 2])) that is different to the input size (torch.Size([1, 512, 2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
C:\Users\filip\anaconda3\envs\ETS_autopylot\Lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([429, 2])) that is different to the input size (torch.Size([1, 429, 2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)


Epoch [1/100], Loss: 0.025158351
Epoch [2/100], Loss: 0.015812611
Epoch [3/100], Loss: 0.011812335
Epoch [4/100], Loss: 0.010481390
Epoch [5/100], Loss: 0.009671867
Epoch [6/100], Loss: 0.009023482
Epoch [7/100], Loss: 0.008487552
Epoch [8/100], Loss: 0.008035387
Epoch [9/100], Loss: 0.007623120
Epoch [10/100], Loss: 0.007266624
Epoch [11/100], Loss: 0.006932987
Epoch [12/100], Loss: 0.006626633
Epoch [13/100], Loss: 0.006342466
Epoch [14/100], Loss: 0.006078806
Epoch [15/100], Loss: 0.005837203
Epoch [16/100], Loss: 0.005614280
Epoch [17/100], Loss: 0.005408155
Epoch [18/100], Loss: 0.005235617
Epoch [19/100], Loss: 0.005076822
Epoch [20/100], Loss: 0.004927600
Epoch [21/100], Loss: 0.004790296
Epoch [22/100], Loss: 0.004669756
Epoch [23/100], Loss: 0.004551331
Epoch [24/100], Loss: 0.004440459
Epoch [25/100], Loss: 0.004336245
Epoch [26/100], Loss: 0.004237755
Epoch [27/100], Loss: 0.004141280
Epoch [28/100], Loss: 0.004050323
Epoch [29/100], Loss: 0.003966182
Epoch [30/100], Loss: 0

In [14]:
# Обучение модели
num_epochs = 50
for epoch in range(num_epochs):
    for batch_x, batch_y in data_loader:  # Итерация по батчам данных
        optimizer.zero_grad()  # Обнуление градиентов

        outputs = model(batch_x.unsqueeze(0))  # Передача входных данных через модель
        loss = criterion(outputs, batch_y)  # Вычисление потерь
        
        l1_loss = l1_penalty(model.parameters())
        l2_loss = l2_penalty(model.parameters())
        total_loss = loss + l1_factor * l1_loss + l2_factor * l2_loss
        
        loss.backward()  # Обратное распространение ошибки
        optimizer.step()  # Обновление весов модели

    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.9f}')

Epoch [1/50], Loss: 0.001737332
Epoch [2/50], Loss: 0.001717864
Epoch [3/50], Loss: 0.001707437
Epoch [4/50], Loss: 0.001696029
Epoch [5/50], Loss: 0.001663349
Epoch [6/50], Loss: 0.001659978
Epoch [7/50], Loss: 0.001650066
Epoch [8/50], Loss: 0.001629126
Epoch [9/50], Loss: 0.001606044
Epoch [10/50], Loss: 0.001599943
Epoch [11/50], Loss: 0.001599209
Epoch [12/50], Loss: 0.001580268
Epoch [13/50], Loss: 0.001592563
Epoch [14/50], Loss: 0.001548767
Epoch [15/50], Loss: 0.001531221
Epoch [16/50], Loss: 0.001583738
Epoch [17/50], Loss: 0.001535583
Epoch [18/50], Loss: 0.001518813
Epoch [19/50], Loss: 0.001492333
Epoch [20/50], Loss: 0.001474130
Epoch [21/50], Loss: 0.001496724
Epoch [22/50], Loss: 0.001511566
Epoch [23/50], Loss: 0.001493446
Epoch [24/50], Loss: 0.001448185
Epoch [25/50], Loss: 0.001447344
Epoch [26/50], Loss: 0.001476168
Epoch [27/50], Loss: 0.001463486
Epoch [28/50], Loss: 0.001444933
Epoch [29/50], Loss: 0.001397904
Epoch [30/50], Loss: 0.001391459
Epoch [31/50], Loss

In [15]:
torch.save(model.state_dict(), 'C:\PycharmProjects\ETS_Autopilot\static\weight_model\weight_wheel_nn_forward_8.pth')